In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

train.head()

Train: (77299, 11)
Test: (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [3]:
def process(df):

    df = df.copy()

    time_parts = df["timestamp"].str.split(":", expand=True)

    df["hour"] = time_parts[0].astype(int)
    df["minute"] = time_parts[1].astype(int)

    df["is_peak_hour"] = (
        df["hour"].isin([7,8,9,17,18,19])
    ).astype(int)

    df.drop(columns=["timestamp"], inplace=True)

    return df


train = process(train)
test = process(test)

print("Feature engineering complete")

Feature engineering complete


In [4]:
categorical_columns = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

for col in categorical_columns:
    train[col] = train[col].fillna("missing").astype(str)
    test[col] = test[col].fillna("missing").astype(str)

train["Temperature"] = train["Temperature"].fillna(
    train["Temperature"].median()
)

test["Temperature"] = test["Temperature"].fillna(
    train["Temperature"].median()
)

In [5]:
print(train.isnull().sum())

Index            0
geohash          0
day              0
demand           0
RoadType         0
NumberofLanes    0
LargeVehicles    0
Landmarks        0
Temperature      0
Weather          0
hour             0
minute           0
is_peak_hour     0
dtype: int64


In [6]:
X = train.drop(columns=["demand"])
y = train["demand"]

print(X.shape)
print(y.shape)

(77299, 12)
(77299,)


In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
cat_features = [
    "geohash",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

model = CatBoostRegressor(
    iterations=2000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 0.1368970	total: 62ms	remaining: 2m 4s
200:	learn: 0.0398827	total: 3.83s	remaining: 34.3s
400:	learn: 0.0361237	total: 7.58s	remaining: 30.2s
600:	learn: 0.0339242	total: 10.6s	remaining: 24.6s
800:	learn: 0.0323973	total: 13.5s	remaining: 20.3s
1000:	learn: 0.0312948	total: 16.5s	remaining: 16.5s
1200:	learn: 0.0304461	total: 19.6s	remaining: 13s
1400:	learn: 0.0297207	total: 22.7s	remaining: 9.71s
1600:	learn: 0.0291288	total: 25.7s	remaining: 6.39s
1800:	learn: 0.0286149	total: 28.8s	remaining: 3.18s
1999:	learn: 0.0281366	total: 31.8s	remaining: 0us


CatBoostRegressor(depth=8, iterations=2000, learning_rate=0.05, loss_function='RMSE', verbose=200)

In [9]:
preds = model.predict(X_val)

score = r2_score(y_val, preds)

print("Validation R2 =", score)

Validation R2 = 0.94525886771951


In [10]:
model = CatBoostRegressor(
    iterations=2000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=200
)

model.fit(
    X,
    y,
    cat_features=cat_features
)

0:	learn: 0.1368946	total: 28.7ms	remaining: 57.4s
200:	learn: 0.0396809	total: 3.75s	remaining: 33.5s
400:	learn: 0.0359959	total: 7.43s	remaining: 29.6s
600:	learn: 0.0337006	total: 10.8s	remaining: 25.2s
800:	learn: 0.0322086	total: 14.1s	remaining: 21.2s
1000:	learn: 0.0310673	total: 17.4s	remaining: 17.4s
1200:	learn: 0.0302750	total: 20.9s	remaining: 13.9s
1400:	learn: 0.0295216	total: 24.4s	remaining: 10.4s
1600:	learn: 0.0289390	total: 28s	remaining: 6.99s
1800:	learn: 0.0284194	total: 31.7s	remaining: 3.5s
1999:	learn: 0.0279542	total: 35.2s	remaining: 0us


CatBoostRegressor(depth=8, iterations=2000, learning_rate=0.05, loss_function='RMSE', verbose=200)

In [11]:
test_predictions = model.predict(test)

print(test_predictions[:10])

[0.01988099 0.01480087 0.0020568  0.01902431 0.03766881 0.00317087
 0.02600211 0.03970349 0.02420291 0.04055635]


In [12]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,Index,demand
0,0,0.019881
1,1,0.014801
2,2,0.002057
3,3,0.019024
4,4,0.037669


In [13]:
sub = pd.read_csv("submission.csv")

print(sub.shape)
print(sub.columns)

(41778, 2)
Index(['Index', 'demand'], dtype='str')
